In [ ]:
!pip uninstall -y transformers peft accelerate -q
!pip install -q \
  transformers==4.41.2 \
  peft==0.11.1 \
  accelerate==0.30.1 \
  datasets \
  scikit-learn \
  sentencepiece \
  safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
output_dir = "/content/drive/MyDrive/CLIP_Prompt_MFND"
os.makedirs(output_dir, exist_ok=True)
print("Save to:", output_dir)

Mounted at /content/drive
Save to: /content/drive/MyDrive/CLIP_Prompt_MFND


In [ ]:
import torch
import numpy as np
import torch.nn as nn
from datasets import load_dataset, Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

dataset = load_dataset("anson-huang/mirage-news")
dataset = dataset.cast_column("image", Image())
dataset = dataset.rename_column("label", "labels")

model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

In [ ]:
PROMPT_LENGTH = 5

def preprocess(examples):
    inputs = processor(
        text=examples["text"],
        images=examples["image"],
        padding="max_length",
        truncation=True,
        max_length=77 - PROMPT_LENGTH
    )
    inputs["labels"] = examples["labels"]
    return inputs

encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names,
    batched=True
)

encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

print("Preprocessing done")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Preprocessing done


In [ ]:
class CLIPPromptTuningForMFND(nn.Module):
    def __init__(self,
                 model_name="openai/clip-vit-base-patch32",
                 num_labels=2,
                 prompt_length=5):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(model_name)
        self.prompt_length = prompt_length

        for param in self.clip.parameters():
            param.requires_grad = False

        self.hidden_dim = self.clip.config.text_config.hidden_size

        self.soft_prompt = nn.Parameter(
            torch.randn(prompt_length, self.hidden_dim)
        )

        self.classifier = nn.Linear(
            self.clip.config.projection_dim * 2,
            num_labels
        )

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        batch_size = input_ids.size(0)
        device = input_ids.device

        token_embeddings = self.clip.text_model.embeddings.token_embedding(input_ids)

        soft_prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        hidden_states = torch.cat([soft_prompt, token_embeddings], dim=1)

        prompt_mask = torch.ones(batch_size, self.prompt_length, device=device)
        attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        seq_length = hidden_states.size(1)
        position_ids = torch.arange(seq_length, device=device).unsqueeze(0)
        position_embeddings = self.clip.text_model.embeddings.position_embedding(position_ids)
        hidden_states = hidden_states + position_embeddings

        causal_attention_mask = torch.full(
            (seq_length, seq_length),
            float("-inf"),
            device=device
        )
        causal_attention_mask = torch.triu(causal_attention_mask, diagonal=1)
        causal_attention_mask = causal_attention_mask.unsqueeze(0).unsqueeze(0)
        causal_attention_mask = causal_attention_mask.expand(batch_size, 1, seq_length, seq_length)

        if attention_mask is not None:
            attention_mask = attention_mask[:, None, None, :]
            attention_mask = attention_mask.expand(batch_size, 1, seq_length, seq_length)

        encoder_outputs = self.clip.text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=attention_mask,
            causal_attention_mask=causal_attention_mask,
        )

        hidden_states = encoder_outputs[0]
        hidden_states = self.clip.text_model.final_layer_norm(hidden_states)

        eos_token_id = self.clip.config.text_config.eos_token_id
        eos_mask = (input_ids == eos_token_id)
        eos_positions = eos_mask.float().argmax(dim=1) + self.prompt_length

        pooled_output = hidden_states[
            torch.arange(batch_size, device=device),
            eos_positions
        ]

        text_features = self.clip.text_projection(pooled_output)

        image_outputs = self.clip.vision_model(pixel_values=pixel_values)
        image_features = self.clip.visual_projection(image_outputs.pooler_output)

        features = torch.cat([text_features, image_features], dim=1)
        logits = self.classifier(features)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

model = CLIPPromptTuningForMFND(prompt_length=PROMPT_LENGTH).to(device)

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Trainable params: 4,610
Total params: 151,281,923
Trainable %: 0.0030%


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()
metrics = trainer.evaluate()
print(metrics)

trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print("✅ Model saved to:", output_dir)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.336400,0.200070,0.938800,0.936356,0.941600,0.938971,0.983807
2,0.184500,0.162738,0.944000,0.954918,0.932000,0.943320,0.988686
3,0.154900,0.143177,0.952400,0.951317,0.953600,0.952457,0.990618
4,0.127400,0.136312,0.952400,0.947036,0.958400,0.952684,0.991367
5,0.121800,0.134090,0.953200,0.949247,0.957600,0.953405,0.991586


{'eval_loss': 0.1340898722410202, 'eval_accuracy': 0.9532, 'eval_precision': 0.9492466296590008, 'eval_recall': 0.9576, 'eval_f1': 0.953405017921147, 'eval_auc': 0.99158592, 'eval_runtime': 36.752, 'eval_samples_per_second': 68.024, 'eval_steps_per_second': 2.15, 'epoch': 5.0}
✅ Model saved to: /content/drive/MyDrive/CLIP_Prompt_MFND


In [ ]:
# ─── 2. Setup ─────────────────────────────────────────────────────────────────
import os, time
import torch
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from safetensors.torch import load_file
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset, Image as HFImage

from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR      = "/content/drive/MyDrive/CLIP_Prompt_MFND"
model_name    = "openai/clip-vit-base-patch32"
PROMPT_LENGTH = 5
device        = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 3. Model class ───────────────────────────────────────────────────────────
class CLIPPromptTuningForMFND(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32",
                 num_labels=2, prompt_length=5):
        super().__init__()
        self.clip          = CLIPModel.from_pretrained(model_name)
        self.prompt_length = prompt_length

        for param in self.clip.parameters():
            param.requires_grad = False

        self.hidden_dim  = self.clip.config.text_config.hidden_size
        self.soft_prompt = nn.Parameter(torch.randn(prompt_length, self.hidden_dim))
        self.classifier  = nn.Linear(self.clip.config.projection_dim * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None,
                pixel_values=None, labels=None):
        batch_size = input_ids.size(0)
        device     = input_ids.device

        token_embeddings = self.clip.text_model.embeddings.token_embedding(input_ids)
        soft_prompt      = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        hidden_states    = torch.cat([soft_prompt, token_embeddings], dim=1)

        prompt_mask    = torch.ones(batch_size, self.prompt_length, device=device)
        attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        seq_length    = hidden_states.size(1)
        position_ids  = torch.arange(seq_length, device=device).unsqueeze(0)
        hidden_states = hidden_states + \
            self.clip.text_model.embeddings.position_embedding(position_ids)

        causal_mask = torch.triu(
            torch.full((seq_length, seq_length), float("-inf"), device=device),
            diagonal=1
        ).unsqueeze(0).unsqueeze(0).expand(batch_size, 1, seq_length, seq_length)

        attention_mask = attention_mask[:, None, None, :] \
            .expand(batch_size, 1, seq_length, seq_length)

        encoder_out   = self.clip.text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=attention_mask,
            causal_attention_mask=causal_mask,
        )
        hidden_states = self.clip.text_model.final_layer_norm(encoder_out[0])

        eos_token_id  = self.clip.config.text_config.eos_token_id
        eos_positions = (input_ids == eos_token_id).float().argmax(dim=1) + self.prompt_length
        eos_positions = eos_positions.clamp(max=seq_length - 1)
        pooled        = hidden_states[torch.arange(batch_size, device=device), eos_positions]
        text_features = self.clip.text_projection(pooled)

        image_out      = self.clip.vision_model(pixel_values=pixel_values)
        image_features = self.clip.visual_projection(image_out.pooler_output)

        fused  = torch.cat([text_features, image_features], dim=1)
        logits = self.classifier(fused)
        loss   = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

# ─── 4. Load model từ Drive ───────────────────────────────────────────────────
print("⏳ Loading CLIP Prompt Tuning...")

# Load processor từ HuggingFace (tránh bug local path của transformers 4.41)
processor = CLIPProcessor.from_pretrained(model_name)

model = CLIPPromptTuningForMFND(model_name, prompt_length=PROMPT_LENGTH).to(device)

safetensor_path = os.path.join(SAVE_DIR, "model.safetensors")
bin_path        = os.path.join(SAVE_DIR, "pytorch_model.bin")

if os.path.exists(safetensor_path):
    state = load_file(safetensor_path, device=device)
    model.load_state_dict(state, strict=False)
    print("✅ Loaded from model.safetensors")
elif os.path.exists(bin_path):
    import functools
    torch.load = functools.partial(torch.load, weights_only=False)
    state = torch.load(bin_path, map_location=device)
    model.load_state_dict(state)
    print("✅ Loaded from pytorch_model.bin")
else:
    raise FileNotFoundError(f"Không tìm thấy weights trong {SAVE_DIR}")

model.eval()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.4f}%)")

# ─── 5. Preprocess + Evaluate helpers ────────────────────────────────────────
def make_test_dataset(split_name):
    ds = load_dataset("anson-huang/mirage-news", split=split_name)
    ds = ds.cast_column("image", HFImage())
    def preprocess(examples):
        inputs = processor(
            text=examples["text"],
            images=examples["image"],
            padding="max_length",
            truncation=True,
            max_length=77 - PROMPT_LENGTH  # giống lúc train
        )
        inputs["labels"] = examples["label"]
        return inputs
    encoded = ds.map(
        preprocess,
        remove_columns=ds.column_names,
        batched=True,
        desc=split_name
    )
    return encoded

def collate_fn(batch):
    result = {}
    for k in batch[0]:
        arr = np.array([b[k] for b in batch])
        t   = torch.tensor(arr)
        if k == "pixel_values":
            t = t.float()
        result[k] = t
    return result

def evaluate_split(dataset):
    loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                pixel_values   = batch["pixel_values"].to(device)
            )
            all_logits.append(out["logits"].cpu())
            all_labels.append(batch["labels"])
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p, "recall": r, "f1": f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── 6. Test Evaluation ───────────────────────────────────────────────────────
test_splits = {
    "ID  test1_nyt_mj":    "test1_nyt_mj",
    "OOD test2_bbc_dalle": "test2_bbc_dalle",
    "OOD test3_cnn_dalle": "test3_cnn_dalle",
    "OOD test4_bbc_sdxl":  "test4_bbc_sdxl",
    "OOD test5_cnn_sdxl":  "test5_cnn_sdxl",
}

print("\n📊 Test Evaluation — CLIP ViT-B/32 Prompt Tuning (MIRAGE-News):")
print(f"{'Split':<26} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>8}")
print("-" * 63)

results, ood_names = {}, []

for name, split in test_splits.items():
    td            = make_test_dataset(split)
    m             = evaluate_split(td)
    results[name] = m
    print(f"{name:<26} {m['accuracy']*100:>6.2f}% {m['precision']*100:>6.2f}% "
          f"{m['recall']*100:>6.2f}% {m['f1']*100:>6.2f}% {m['auc']:>8.5f}")
    if name.startswith("OOD"):
        ood_names.append(name)

print("-" * 63)
ood_avg = {m: np.mean([results[n][m] for n in ood_names])
           for m in ["accuracy", "precision", "recall", "f1", "auc"]}
print(f"{'OOD Average':<26} {ood_avg['accuracy']*100:>6.2f}% {ood_avg['precision']*100:>6.2f}% "
      f"{ood_avg['recall']*100:>6.2f}% {ood_avg['f1']*100:>6.2f}% {ood_avg['auc']:>8.5f}")

# ─── 7. Latency + VRAM ───────────────────────────────────────────────────────
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inp = processor(text=dummy_text, images=dummy_image,
                truncation=True, padding="max_length",
                max_length=77 - PROMPT_LENGTH, return_tensors="pt")
input_ids      = inp["input_ids"].to(device)
attention_mask = inp["attention_mask"].to(device)
pixel_values   = inp["pixel_values"].float().to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(input_ids, attention_mask, pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids, attention_mask, pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0
print(f"\n{'='*40}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)     : {vram:.1f} MB")
print(f"{'='*40}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
⏳ Loading CLIP Prompt Tuning...
✅ Loaded from model.safetensors
Total params    : 151,281,923
Trainable params: 4,610 (0.0030%)

📊 Test Evaluation — CLIP ViT-B/32 Prompt Tuning (MIRAGE-News):
Split                          Acc    Prec     Rec      F1      AUC
---------------------------------------------------------------


test1_nyt_mj:   0%|          | 0/500 [00:00<?, ? examples/s]

ID  test1_nyt_mj            96.40%  96.77%  96.00%  96.39%  0.99518


test2_bbc_dalle:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD test2_bbc_dalle         65.80%  66.53%  63.60%  65.03%  0.72437


...

OOD test3_cnn_dalle         67.40%  67.76%  66.40%  67.07%  0.76030


test4_bbc_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD test4_bbc_sdxl          75.00%  70.90%  84.80%  77.23%  0.81597


test5_cnn_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD test5_cnn_sdxl          77.80%  73.72%  86.40%  79.56%  0.87509
---------------------------------------------------------------
OOD Average                 71.50%  69.73%  75.30%  72.22%  0.79393

Total params    : 151,281,923
Trainable params: 4,610
Latency median  : 17.17 ms
VRAM (peak)     : 1756.8 MB
